# 02 — Training Curves

Visualise fine-tuning convergence: loss curves, per-epoch accuracy and F1.

**Run after:** `make train`

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

state = json.loads((Path("models/classifier") / "trainer_state.json").read_text())
history = pd.DataFrame(state["log_history"])
train_log = history.dropna(subset=["loss"])
eval_log  = history.dropna(subset=["eval_loss"])
print(f"Epochs logged: {eval_log['epoch'].max():.0f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
train_log.plot(x="epoch", y="loss",          ax=axes[0], title="Train loss",   legend=False)
eval_log.plot( x="epoch", y="eval_loss",     ax=axes[1], title="Val loss",     legend=False)
eval_log.plot( x="epoch", y="eval_f1_macro", ax=axes[2], title="Val F1 macro", legend=False)
plt.tight_layout()
plt.savefig("notebooks/training_curves.png", dpi=150)
plt.show()

In [ ]:
best = eval_log.sort_values("eval_f1_macro", ascending=False).iloc[0]
print(f"Best epoch:      {best['epoch']:.0f}")
print(f"Val accuracy:    {best['eval_accuracy']:.4f}")
print(f"Val F1 macro:    {best['eval_f1_macro']:.4f}")
print(f"Val F1 weighted: {best['eval_f1_weighted']:.4f}")

## Hyperparameter comparison

Compare key training settings and their effect on final validation metrics. Re-run with different `configs/training.yaml` values and record results below.

In [ ]:
import yaml

config = yaml.safe_load(Path("configs/training.yaml").read_text())

# Current run summary
current = {
    "model":         config["model_name"],
    "lr":            config["learning_rate"],
    "batch_size":    config["batch_size"],
    "epochs_run":    int(eval_log["epoch"].max()),
    "warmup_ratio":  config["warmup_ratio"],
    "weight_decay":  config["weight_decay"],
    "best_f1_macro": float(best["eval_f1_macro"]),
    "best_accuracy": float(best["eval_accuracy"]),
}

# Append to comparison log (add rows manually for different runs)
comparison = pd.DataFrame([current])
print(comparison.to_string(index=False))